# Rapid Lightning v1 — production training pipeline

The recipe. Takes a tabular classification dataset, produces a 19 KB self-contained
`.so` predictor with sub-microsecond CPU latency. **This notebook is the reference
for the inference-service training endpoint.**

> Ship config: vanilla v1 (`class_weight=False`, accuracy-mode fitness, m=10,
> pop=200, gens=40, depth=6). v1.1 / balanced-mode is in the lab folder for
> v2 worlds-of-fitness work; not in production.

What this notebook does:

1. Embed the four source files (`gp_ensemble`, `jit_kernel`, `bridge`, `lightning_aot`)
2. Load a dataset (UCI Pima as the demo; replace with your data)
3. Train the GP ensemble + LR head with the v1 ship config
4. AOT-compile to a self-contained `.so` via `lightning_aot`
5. Verify AOT predictions match the production JIT path bit-for-bit
6. Measure pure-C single-sample latency with `clock_gettime`
7. Bundle `.so` + metadata for deployment


## 1. Environment


In [ ]:
import sys, platform, subprocess
print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())
try:
    out = subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0]
    print('gcc      :', out)
except FileNotFoundError:
    print('gcc      : NOT FOUND  (apt install build-essential)')


In [ ]:
# numba 0.60 + llvmlite 0.43 keep us compatible with lleaves if we ever need it
# in the same env. For pure production training we only need numba/numpy/sklearn/torch.
%pip install -q --quiet 'numba==0.60.0' 'llvmlite==0.43.0'
%pip install -q numpy scikit-learn pandas torch


In [ ]:
import numpy as np, numba
print('numpy', np.__version__, '| numba', numba.__version__)


## 2. Source files

The four embedded files compose the production pipeline:

- `gp_ensemble.py` — GP evolution, fitness, ensemble training
- `jit_kernel.py`  — Numba JIT inference (development / verification path)
- `bridge.py`      — `GPNode` → bytecode flattening
- `lightning_aot.py` — AOT C99 emitter, ctypes loader, in-C timing

Run all four `%%writefile` cells, then sanity-check.


In [ ]:
%%writefile gp_ensemble.py
#!/usr/bin/env python3
"""
FMA Evolution Lab v2.4 -- Profiling the Winning GPE
=====================================================

v2.3 established the result. v2.4 takes the winning method outside
to run around in the yard:

- Profile inference latency / throughput / size for both winners:
    GPE-LR        (compute champion -- lean, ~0 meta-params)
    GPE-XGB-Aug   (accuracy champion -- GP + 200-tree XGB)
- Try Fast and Powerful variants to map the cost/accuracy frontier.
- Add Tier 5 (UCI Bank Marketing) for a third tabular dataset.

Reproducibility: Tiers 1-4 unchanged from v2.3 with same arguments.

Usage:
    python gp_ensemble.py --tier 3                  # Adult
    python gp_ensemble.py --tier 5                  # Bank Marketing (new)
    python profile_winner.py --tier 3               # profile both winners
    python run_fast.py --tier 3                     # 5-member fast variant
    python run_powerful.py --tier 3                 # 100-member ceiling test
"""

import argparse
import json
import math
import random
import time
from collections import Counter
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    tier: int = 1
    num_members: int = 25
    boost: bool = True

    # Per-member GP
    pop_size: int = 200
    num_islands: int = 4
    max_depth: int = 8
    gens: int = 80
    migrate_every: int = 12
    migrate_count: int = 3
    tournament_size: int = 7
    crossover_rate: float = 0.85
    mutation_rate: float = 0.20
    parsimony: float = 0.0001
    elite_pct: float = 0.05
    fresh_pct: float = 0.03

    # Boosting
    boost_weight: float = 3.0
    boost_after: int = 3

    # Class-weighted fitness (v2.3): multiply sample weights by inverse
    # class prevalence so GP evolution optimizes balanced accuracy instead
    # of raw accuracy. Helpful on imbalanced datasets (e.g. Adult 76/24).
    class_weight: bool = False

    # Tier 4 (covertype) can be large (~580K rows). Subsample to keep
    # sweep experiments tractable. Set to 0 for full dataset.
    covtype_subsample: int = 50000

    # Meta-learner
    meta_lr: float = 0.01
    meta_epochs: int = 200
    meta_hidden: int = 64
    meta_batch_size: int = 64

    # NN baseline
    nn_hidden: int = 128
    nn_lr: float = 0.01
    nn_epochs: int = 100
    nn_batch_size: int = 128

    device: str = 'cuda'
    seed: int = 42


# ============================================================
# DATASET
# ============================================================

def load_dataset(tier, device, cfg=None):
    if tier == 1:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        digits = load_digits()
        X, y = digits.data / 16.0, digits.target
        X_trainfull, X_te, y_trainfull, y_te = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)
        # Val slice carved from train. Test stays untouched.
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_trainfull, y_trainfull, test_size=0.15, random_state=43,
            stratify=y_trainfull)
        return {
            'X_train': torch.tensor(X_tr, dtype=torch.float32, device=device),
            'y_train': torch.tensor(y_tr, dtype=torch.long, device=device),
            'X_val': torch.tensor(X_val, dtype=torch.float32, device=device),
            'y_val': torch.tensor(y_val, dtype=torch.long, device=device),
            'X_test': torch.tensor(X_te, dtype=torch.float32, device=device),
            'y_test': torch.tensor(y_te, dtype=torch.long, device=device),
            'input_size': 64, 'num_classes': 10, 'name': 'sklearn-digits-8x8',
        }
    elif tier == 2:
        from torchvision import datasets
        from sklearn.model_selection import train_test_split
        # Download raw MNIST (28x28). We downsample to 14x14 ourselves so the
        # pipeline is deterministic and doesn't depend on torchvision's
        # transform pipeline firing at the right time (it only runs in
        # __getitem__, not on .data, which is a footgun).
        train_ds = datasets.MNIST('./data', train=True, download=True)
        test_ds = datasets.MNIST('./data', train=False, download=True)

        def to_14x14(raw_uint8):
            """(N, 28, 28) uint8 -> (N, 196) float32 numpy in [0,1]"""
            x = raw_uint8.float().unsqueeze(1) / 255.0            # (N, 1, 28, 28)
            x = F.interpolate(x, size=(14, 14), mode='bilinear',
                              align_corners=False)                 # (N, 1, 14, 14)
            return x.view(-1, 196).numpy()

        X_trainfull = to_14x14(train_ds.data)
        y_trainfull = train_ds.targets.numpy()
        X_test_np = to_14x14(test_ds.data)
        y_test_np = test_ds.targets.numpy()

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_trainfull, y_trainfull, test_size=5000, random_state=43,
            stratify=y_trainfull)
        return {
            'X_train': torch.tensor(X_tr, dtype=torch.float32, device=device),
            'y_train': torch.tensor(y_tr, dtype=torch.long, device=device),
            'X_val': torch.tensor(X_val, dtype=torch.float32, device=device),
            'y_val': torch.tensor(y_val, dtype=torch.long, device=device),
            'X_test': torch.tensor(X_test_np, dtype=torch.float32, device=device),
            'y_test': torch.tensor(y_test_np, dtype=torch.long, device=device),
            'input_size': 196, 'num_classes': 10, 'name': 'MNIST-14x14',
        }
    elif tier == 3:
        # UCI Adult: binary income classification, ~48K samples, 14 raw features
        # (6 numeric + 8 categorical -> ~100 features after one-hot).
        # This is where GPEs will face XGBoost on its home turf.
        import numpy as np
        import pandas as pd
        from sklearn.datasets import fetch_openml
        from sklearn.preprocessing import StandardScaler, OneHotEncoder
        from sklearn.compose import ColumnTransformer
        from sklearn.model_selection import train_test_split

        print("  Fetching UCI Adult from OpenML (cached after first run)...")
        adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
        X_df = adult.data.copy()
        y_ser = adult.target

        # Target: '>50K' -> 1, '<=50K' -> 0
        y = (y_ser.astype(str).str.replace('.', '', regex=False) == '>50K').astype(int).values

        # '?' sentinel -> NaN, then impute
        X_df = X_df.replace('?', np.nan)
        cat_cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
        num_cols = X_df.select_dtypes(include=['number']).columns.tolist()
        for c in cat_cols:
            X_df[c] = X_df[c].fillna(X_df[c].mode().iloc[0])
        for c in num_cols:
            X_df[c] = X_df[c].fillna(X_df[c].median())

        # Split FIRST, then fit preprocessor on train only (no leakage)
        X_trfull_df, X_test_df, y_trfull, y_test_np = train_test_split(
            X_df, y, test_size=0.2, random_state=42, stratify=y)

        preproc = ColumnTransformer([
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ])
        preproc.fit(X_trfull_df)
        X_trfull = preproc.transform(X_trfull_df).astype('float32')
        X_test_np = preproc.transform(X_test_df).astype('float32')

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_trfull, y_trfull, test_size=0.15, random_state=43, stratify=y_trfull)

        input_size = X_tr.shape[1]
        pos_rate = y_trfull.mean()
        print(f"  Features: {len(num_cols)} numeric + {len(cat_cols)} categorical"
              f" -> {input_size} after one-hot encoding")
        print(f"  Class balance: {pos_rate:.1%} positive (>50K), "
              f"{1-pos_rate:.1%} negative (<=50K)")
        return {
            'X_train': torch.tensor(X_tr, dtype=torch.float32, device=device),
            'y_train': torch.tensor(y_tr, dtype=torch.long, device=device),
            'X_val': torch.tensor(X_val, dtype=torch.float32, device=device),
            'y_val': torch.tensor(y_val, dtype=torch.long, device=device),
            'X_test': torch.tensor(X_test_np, dtype=torch.float32, device=device),
            'y_test': torch.tensor(y_test_np, dtype=torch.long, device=device),
            'input_size': input_size, 'num_classes': 2, 'name': 'UCI-Adult',
        }
    elif tier == 4:
        # Forest Covertype: 7-class forest cover type from geographic features.
        # 581k rows, 54 numeric features (already numeric, no one-hot needed).
        # Subsample by default to keep sweep experiments tractable on 4090.
        import numpy as np
        from sklearn.datasets import fetch_covtype
        from sklearn.preprocessing import StandardScaler
        from sklearn.model_selection import train_test_split

        print("  Fetching Forest Covertype from sklearn (cached after first run)...")
        cov = fetch_covtype()
        X_all, y_all = cov.data.astype('float32'), cov.target - 1  # 1..7 -> 0..6

        # Subsample (stratified) if configured
        sub = (cfg.covtype_subsample if cfg is not None else 50000)
        if sub and sub < len(X_all):
            X_all, _, y_all, _ = train_test_split(
                X_all, y_all, train_size=sub, random_state=42, stratify=y_all)
            print(f"  Stratified subsample to {sub:,} rows (set --covtype-subsample 0 for full 581k)")
        else:
            print(f"  Using full covertype: {len(X_all):,} rows")

        # Split before scaling (no leakage)
        X_trfull, X_test_np, y_trfull, y_test_np = train_test_split(
            X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
        scaler = StandardScaler()
        X_trfull = scaler.fit_transform(X_trfull).astype('float32')
        X_test_np = scaler.transform(X_test_np).astype('float32')

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_trfull, y_trfull, test_size=0.15, random_state=43, stratify=y_trfull)

        input_size = X_tr.shape[1]
        print(f"  Features: {input_size} numeric (standardized)")
        class_dist = np.bincount(y_trfull) / len(y_trfull)
        print(f"  Class distribution: {[f'{p:.1%}' for p in class_dist]}")
        return {
            'X_train': torch.tensor(X_tr, dtype=torch.float32, device=device),
            'y_train': torch.tensor(y_tr, dtype=torch.long, device=device),
            'X_val': torch.tensor(X_val, dtype=torch.float32, device=device),
            'y_val': torch.tensor(y_val, dtype=torch.long, device=device),
            'X_test': torch.tensor(X_test_np, dtype=torch.float32, device=device),
            'y_test': torch.tensor(y_test_np, dtype=torch.long, device=device),
            'input_size': input_size, 'num_classes': 7, 'name': 'Forest-Covertype',
        }
    elif tier == 5:
        # UCI Bank Marketing: binary classification, ~45K rows.
        # Predict whether client subscribes to a term deposit after marketing call.
        # Mix of numeric and categorical, like Adult but a different domain.
        import numpy as np
        import pandas as pd
        from sklearn.datasets import fetch_openml
        from sklearn.preprocessing import StandardScaler, OneHotEncoder
        from sklearn.compose import ColumnTransformer
        from sklearn.model_selection import train_test_split

        print("  Fetching UCI Bank Marketing from OpenML (cached after first run)...")
        bank = fetch_openml('bank-marketing', version=1, as_frame=True, parser='auto')
        X_df = bank.data.copy()
        y_ser = bank.target

        # Target: '2' or 'yes' = subscribed
        y = (y_ser.astype(str).isin(['2', 'yes'])).astype(int).values

        cat_cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
        num_cols = X_df.select_dtypes(include=['number']).columns.tolist()
        for c in cat_cols:
            X_df[c] = X_df[c].fillna(X_df[c].mode().iloc[0])
        for c in num_cols:
            X_df[c] = X_df[c].fillna(X_df[c].median())

        X_trfull_df, X_test_df, y_trfull, y_test_np = train_test_split(
            X_df, y, test_size=0.2, random_state=42, stratify=y)

        preproc = ColumnTransformer([
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ])
        preproc.fit(X_trfull_df)
        X_trfull = preproc.transform(X_trfull_df).astype('float32')
        X_test_np = preproc.transform(X_test_df).astype('float32')

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_trfull, y_trfull, test_size=0.15, random_state=43, stratify=y_trfull)

        input_size = X_tr.shape[1]
        pos_rate = y_trfull.mean()
        print(f"  Features: {len(num_cols)} numeric + {len(cat_cols)} categorical"
              f" -> {input_size} after one-hot encoding")
        print(f"  Class balance: {pos_rate:.1%} positive (subscribed), "
              f"{1-pos_rate:.1%} negative")
        return {
            'X_train': torch.tensor(X_tr, dtype=torch.float32, device=device),
            'y_train': torch.tensor(y_tr, dtype=torch.long, device=device),
            'X_val': torch.tensor(X_val, dtype=torch.float32, device=device),
            'y_val': torch.tensor(y_val, dtype=torch.long, device=device),
            'X_test': torch.tensor(X_test_np, dtype=torch.float32, device=device),
            'y_test': torch.tensor(y_test_np, dtype=torch.long, device=device),
            'input_size': input_size, 'num_classes': 2, 'name': 'UCI-Bank-Marketing',
        }
    raise ValueError(f"Unknown tier: {tier}")


# ============================================================
# GP ENGINE (same as ensemble)
# ============================================================

OPS = {
    'fma': {'arity': 3}, 'add': {'arity': 2}, 'mul': {'arity': 2},
    'sub': {'arity': 2}, 'max': {'arity': 2}, 'min': {'arity': 2},
    'neg': {'arity': 1}, 'abs': {'arity': 1}, 'clamp': {'arity': 1},
    'lt':  {'arity': 2},  # less-than: returns 1.0 if a<b else 0.0 (for threshold-style rules)
}
OP_KEYS = list(OPS.keys())

# Tier-aware primitive subsets so existing Tier 1/2 runs are reproducible.
# Tier 1 & 2 keep the original op set from v2.0 (no 'lt').
# Tier 3 (tabular) adds 'lt' for threshold-style decision boundaries.
TIER_OP_SUBSETS = {
    1: ['fma', 'add', 'mul', 'sub', 'max', 'min', 'neg', 'abs', 'clamp'],
    2: ['fma', 'add', 'mul', 'sub', 'max', 'min', 'neg', 'abs', 'clamp'],
    3: ['fma', 'add', 'mul', 'sub', 'max', 'min', 'neg', 'abs', 'clamp', 'lt'],
    4: ['fma', 'add', 'mul', 'sub', 'max', 'min', 'neg', 'abs', 'clamp', 'lt'],
    5: ['fma', 'add', 'mul', 'sub', 'max', 'min', 'neg', 'abs', 'clamp', 'lt'],
}
# Module-level active set; set by set_active_ops(tier) at the start of run().
ACTIVE_OP_KEYS = TIER_OP_SUBSETS[1]

def set_active_ops(tier):
    global ACTIVE_OP_KEYS
    ACTIVE_OP_KEYS = TIER_OP_SUBSETS.get(tier, OP_KEYS)


class GPNode:
    __slots__ = ['ntype', 'value', 'children']
    def __init__(self, ntype, value, children=None):
        self.ntype = ntype; self.value = value; self.children = children or []
    def evaluate(self, inputs):
        if self.ntype == 'input': return inputs[:, self.value % inputs.shape[1]]
        if self.ntype == 'const':
            return torch.full((inputs.shape[0],), self.value, device=inputs.device, dtype=inputs.dtype)
        args = [c.evaluate(inputs) for c in self.children]
        op = self.value
        if   op == 'fma':   r = args[0] * args[1] + args[2]
        elif op == 'add':   r = args[0] + args[1]
        elif op == 'mul':   r = args[0] * args[1]
        elif op == 'sub':   r = args[0] - args[1]
        elif op == 'max':   r = torch.maximum(args[0], args[1])
        elif op == 'min':   r = torch.minimum(args[0], args[1])
        elif op == 'neg':   r = -args[0]
        elif op == 'abs':   r = torch.abs(args[0])
        elif op == 'clamp': r = torch.clamp(args[0], 0.0, 1.0)
        elif op == 'lt':    r = (args[0] < args[1]).to(args[0].dtype)
        else:               r = args[0]
        return torch.clamp(r, -1e6, 1e6)
    def size(self):
        return 1 + sum(c.size() for c in self.children)
    def depth(self):
        return 1 if not self.children else 1 + max(c.depth() for c in self.children)
    def to_str(self):
        if self.ntype == 'input': return f'x{self.value}'
        if self.ntype == 'const': return f'{self.value:.2f}'
        return f'{self.value}({", ".join(c.to_str() for c in self.children)})'
    def clone(self):
        return GPNode(self.ntype, self.value, [c.clone() for c in self.children])
    def all_nodes(self):
        nodes = [self]
        for c in self.children: nodes.extend(c.all_nodes())
        return nodes


class GPIndividual:
    __slots__ = ['trees', 'fitness', 'train_acc']
    def __init__(self, trees, fitness=0.0, train_acc=0.0):
        self.trees = trees; self.fitness = fitness; self.train_acc = train_acc
    def classify(self, inputs):
        scores = torch.stack([t.evaluate(inputs) for t in self.trees], dim=1)
        return scores.argmax(dim=1)
    def get_scores(self, inputs):
        """Return raw per-class scores (not argmax) for stacking."""
        return torch.stack([t.evaluate(inputs) for t in self.trees], dim=1)
    def total_size(self):
        return sum(t.size() for t in self.trees)
    def clone(self):
        return GPIndividual([t.clone() for t in self.trees], self.fitness, self.train_acc)
    def used_features(self, input_size):
        feats = set()
        for t in self.trees:
            for n in t.all_nodes():
                if n.ntype == 'input': feats.add(n.value % input_size)
        return feats


def random_tree(md, ins, d=0):
    if d >= md or (d > 1 and random.random() < 0.3):
        if random.random() < 0.7: return GPNode('input', random.randint(0, ins - 1))
        return GPNode('const', random.uniform(-2, 2))
    op = random.choice(ACTIVE_OP_KEYS)
    return GPNode('op', op, [random_tree(md, ins, d+1) for _ in range(OPS[op]['arity'])])

def random_individual(nc, md, ins):
    return GPIndividual([random_tree(md, ins) for _ in range(nc)])

def mutate(ind, md, ins):
    m = ind.clone(); ti = random.randint(0, len(m.trees)-1)
    nodes = m.trees[ti].all_nodes()
    if len(nodes) <= 1 or random.random() < 0.3: m.trees[ti] = random_tree(md, ins)
    else:
        t = random.choice(nodes)
        if t.ntype == 'const': t.value += random.gauss(0, 0.5)
        elif t.ntype == 'input': t.value = random.randint(0, ins-1)
        else:
            n = random_tree(min(3, md-1), ins)
            t.ntype, t.value, t.children = n.ntype, n.value, n.children
    return m

def xover(p1, p2, md, ins):
    ch = p1.clone(); ti = random.randint(0, len(ch.trees)-1)
    n1, n2 = ch.trees[ti].all_nodes(), p2.trees[ti].all_nodes()
    if len(n1) > 1 and len(n2) > 1:
        tgt = random.choice(n1); don = random.choice(n2).clone()
        tgt.ntype, tgt.value, tgt.children = don.ntype, don.value, don.children
    if ch.trees[ti].depth() > md + 2: ch.trees[ti] = random_tree(md, ins)
    return ch

def tsel(pop, sz=5):
    best = None
    for _ in range(sz):
        c = random.choice(pop)
        if best is None or c.fitness > best.fitness: best = c
    return best

@torch.no_grad()
def eval_pop(pop, X, y, pars, weights=None):
    for ind in pop:
        try:
            preds = ind.classify(X)
            correct = (preds == y)
            if weights is not None:
                ind.train_acc = (correct.float() * weights).sum().item() / weights.sum().item()
            else:
                ind.train_acc = correct.float().mean().item()
            ind.fitness = ind.train_acc - ind.total_size() * pars
        except (RuntimeError, RecursionError):
            ind.fitness = 0.0; ind.train_acc = 0.0

@torch.no_grad()
def get_accuracy(ind, X, y):
    try: return (ind.classify(X) == y).float().mean().item()
    except: return 0.0

def evolve_member(X_train, y_train, cfg, weights=None, member_num=0):
    ins = X_train.shape[1]; nc = y_train.max().item() + 1
    random.seed(cfg.seed + member_num * 1000)

    islands = []
    ppi = cfg.pop_size // cfg.num_islands
    for i in range(cfg.num_islands):
        islands.append([random_individual(nc, cfg.max_depth, ins) for _ in range(ppi)])

    best_ind = None; best_acc = 0.0

    for gen in range(1, cfg.gens + 1):
        for i in range(len(islands)):
            pop = islands[i]
            eval_pop(pop, X_train, y_train, cfg.parsimony, weights)
            pop.sort(key=lambda x: x.fitness, reverse=True)
            raw_acc = get_accuracy(pop[0], X_train, y_train)
            if raw_acc > best_acc: best_acc = raw_acc; best_ind = pop[0].clone()

            new_pop = []
            ec = max(2, int(len(pop) * cfg.elite_pct))
            for j in range(ec): new_pop.append(pop[j].clone())
            while len(new_pop) < len(pop):
                p1, p2 = tsel(pop, cfg.tournament_size), tsel(pop, cfg.tournament_size)
                child = xover(p1, p2, cfg.max_depth, ins) if random.random() < cfg.crossover_rate else p1.clone()
                if random.random() < cfg.mutation_rate: child = mutate(child, cfg.max_depth, ins)
                new_pop.append(child)
            fc = int(len(pop) * cfg.fresh_pct)
            for j in range(fc):
                ri = len(pop) - 1 - j
                if ri >= ec: new_pop[ri] = random_individual(nc, cfg.max_depth, ins)
            islands[i] = new_pop

        if gen % cfg.migrate_every == 0:
            ni = len(islands)
            for i in range(ni):
                src = sorted(islands[i], key=lambda x: x.fitness, reverse=True)
                tgt = sorted(islands[(i+1)%ni], key=lambda x: x.fitness)
                for m in range(min(cfg.migrate_count, len(src), len(tgt))):
                    tgt[m] = src[m].clone()
                islands[(i+1)%ni] = tgt

    return best_ind


# ============================================================
# STACKING: EXTRACT FEATURES FROM GP MEMBERS
# ============================================================

@torch.no_grad()
def extract_stacked_features(members, X):
    """
    Run all GP members on X, collect their raw score vectors.
    Returns tensor of shape (n_samples, n_members * n_classes)
    """
    all_scores = []
    for member in members:
        try:
            scores = member.get_scores(X)  # (n_samples, n_classes)
            # Replace NaN/Inf with 0
            scores = torch.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=-1e6)
            all_scores.append(scores)
        except (RuntimeError, RecursionError):
            nc = len(member.trees)
            all_scores.append(torch.zeros(X.shape[0], nc, device=X.device))

    return torch.cat(all_scores, dim=1)  # (n_samples, n_members * n_classes)


# ============================================================
# META-LEARNERS
# ============================================================

def majority_vote(members, X, num_classes):
    """Standard majority vote baseline."""
    n = X.shape[0]
    all_preds = []
    for m in members:
        try: all_preds.append(m.classify(X))
        except: pass
    if not all_preds:
        return torch.zeros(n, dtype=torch.long, device=X.device)
    pred_stack = torch.stack(all_preds, dim=0)
    result = torch.zeros(n, dtype=torch.long, device=X.device)
    for i in range(n):
        votes = pred_stack[:, i].cpu().tolist()
        result[i] = Counter(votes).most_common(1)[0][0]
    return result


def train_logistic_meta(stacked_train, y_train, stacked_test, y_test):
    """Logistic Regression meta-learner (sklearn)."""
    X_tr = stacked_train.cpu().numpy()
    X_te = stacked_test.cpu().numpy()
    y_tr = y_train.cpu().numpy()
    y_te = y_test.cpu().numpy()

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    lr = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
    lr.fit(X_tr_s, y_tr)

    train_acc = lr.score(X_tr_s, y_tr)
    test_acc = lr.score(X_te_s, y_te)

    return train_acc, test_acc, lr, scaler


class MetaMLP(nn.Module):
    """Small MLP meta-learner that learns to combine GP member scores."""
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)


def train_mlp_meta(stacked_train, y_train, stacked_val, y_val, stacked_test, y_test, cfg, device):
    """Train a small MLP meta-learner on stacked GP features.
    Early stopping uses VAL. Test is read ONCE at the end from the val-selected weights.
    """
    input_dim = stacked_train.shape[1]
    num_classes = y_train.max().item() + 1

    # Normalize using TRAIN stats only
    mean = stacked_train.mean(dim=0)
    std = stacked_train.std(dim=0) + 1e-8
    X_tr = (stacked_train - mean) / std
    X_val = (stacked_val - mean) / std
    X_te = (stacked_test - mean) / std

    model = MetaMLP(input_dim, cfg.meta_hidden, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.meta_lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_val = 0.0
    best_state = None

    for epoch in range(1, cfg.meta_epochs + 1):
        model.train()
        idx = torch.randperm(X_tr.shape[0], device=device)
        for start in range(0, X_tr.shape[0], cfg.meta_batch_size):
            batch_idx = idx[start:start+cfg.meta_batch_size]
            optimizer.zero_grad()
            loss = criterion(model(X_tr[batch_idx]), y_train[batch_idx])
            loss.backward()
            optimizer.step()

        if epoch % 20 == 0 or epoch == cfg.meta_epochs:
            model.eval()
            with torch.no_grad():
                val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
            if val_acc > best_val:
                best_val = val_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Restore best-by-val weights
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        train_acc = (model(X_tr).argmax(1) == y_train).float().mean().item()
        val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        test_acc = (model(X_te).argmax(1) == y_test).float().mean().item()

    meta_params = sum(p.numel() for p in model.parameters())
    return train_acc, val_acc, test_acc, model, mean, std, meta_params


def train_xgb_meta(stacked_train, y_train, stacked_val, y_val, stacked_test, y_test, cfg):
    """XGBoost meta-learner on stacked GP features.
    Uses val for early stopping; test read once at the end.
    Returns None if xgboost not installed."""
    try:
        from xgboost import XGBClassifier
    except ImportError:
        return None

    X_tr = stacked_train.cpu().numpy()
    X_va = stacked_val.cpu().numpy()
    X_te = stacked_test.cpu().numpy()
    y_tr = y_train.cpu().numpy()
    y_va = y_val.cpu().numpy()
    y_te = y_test.cpu().numpy()

    xgb = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        early_stopping_rounds=20,
        eval_metric='mlogloss' if len(set(y_tr.tolist())) > 2 else 'logloss',
        tree_method='hist',
        random_state=cfg.seed,
        n_jobs=-1,
        verbosity=0,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    train_acc = xgb.score(X_tr, y_tr)
    val_acc   = xgb.score(X_va, y_va)
    test_acc  = xgb.score(X_te, y_te)
    test_preds = xgb.predict(X_te)

    used_trees = xgb.best_iteration + 1 if hasattr(xgb, 'best_iteration') else xgb.n_estimators
    approx_nodes = used_trees * (2 ** 6)  # upper bound
    return {
        'train': train_acc, 'val': val_acc, 'test': test_acc,
        'trees_used': used_trees, 'approx_nodes_ub': approx_nodes,
        'test_preds': test_preds,
    }


def train_xgb_augmented_meta(X_train_raw, stacked_train, y_train,
                              X_val_raw, stacked_val, y_val,
                              X_test_raw, stacked_test, y_test, cfg):
    """XGBoost meta-learner on concat(raw features, stacked GP features).
    Tests whether GP features are COMPLEMENTARY to raw features.
    Returns None if xgboost not installed."""
    try:
        from xgboost import XGBClassifier
    except ImportError:
        return None

    import numpy as np
    X_tr = np.concatenate([X_train_raw.cpu().numpy(), stacked_train.cpu().numpy()], axis=1)
    X_va = np.concatenate([X_val_raw.cpu().numpy(),   stacked_val.cpu().numpy()],   axis=1)
    X_te = np.concatenate([X_test_raw.cpu().numpy(),  stacked_test.cpu().numpy()],  axis=1)
    y_tr = y_train.cpu().numpy()
    y_va = y_val.cpu().numpy()
    y_te = y_test.cpu().numpy()

    xgb = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        early_stopping_rounds=20,
        eval_metric='mlogloss' if len(set(y_tr.tolist())) > 2 else 'logloss',
        tree_method='hist',
        random_state=cfg.seed,
        n_jobs=-1,
        verbosity=0,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    train_acc = xgb.score(X_tr, y_tr)
    val_acc   = xgb.score(X_va, y_va)
    test_acc  = xgb.score(X_te, y_te)
    test_preds = xgb.predict(X_te)

    used_trees = xgb.best_iteration + 1 if hasattr(xgb, 'best_iteration') else xgb.n_estimators
    fi = xgb.feature_importances_
    n_raw = X_train_raw.shape[1]
    raw_importance = float(fi[:n_raw].sum())
    stacked_importance = float(fi[n_raw:].sum())
    return {
        'train': train_acc, 'val': val_acc, 'test': test_acc,
        'trees_used': used_trees,
        'input_dim': X_tr.shape[1],
        'raw_importance': raw_importance,
        'stacked_importance': stacked_importance,
        'test_preds': test_preds,
    }


def weighted_vote_meta(stacked_train, y_train, stacked_test, y_test, num_members, num_classes):
    """Learn per-member weights via simple optimization."""
    device = stacked_train.device

    # Per-member accuracy on training set -> use as vote weight
    member_weights = torch.ones(num_members, device=device)

    # Evaluate each member individually
    for m in range(num_members):
        start = m * num_classes
        end = start + num_classes
        member_scores = stacked_train[:, start:end]
        member_preds = member_scores.argmax(dim=1)
        member_acc = (member_preds == y_train).float().mean().item()
        member_weights[m] = max(0.01, member_acc)  # Weight by accuracy

    # Weighted score aggregation
    def predict_weighted(stacked, weights):
        n = stacked.shape[0]
        combined = torch.zeros(n, num_classes, device=device)
        for m in range(num_members):
            start = m * num_classes
            end = start + num_classes
            combined += stacked[:, start:end] * weights[m]
        return combined.argmax(dim=1)

    train_preds = predict_weighted(stacked_train, member_weights)
    test_preds = predict_weighted(stacked_test, member_weights)

    train_acc = (train_preds == y_train).float().mean().item()
    test_acc = (test_preds == y_test).float().mean().item()

    return train_acc, test_acc, member_weights


# ============================================================
# NN BASELINE
# ============================================================

class BaselineNN(nn.Module):
    def __init__(self, ins, hs, nc):
        super().__init__()
        self.fc1 = nn.Linear(ins, hs)
        self.fc2 = nn.Linear(hs, nc)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

def train_baseline(ds, cfg):
    model = BaselineNN(ds['input_size'], cfg.nn_hidden, ds['num_classes']).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.nn_lr)
    crit = nn.CrossEntropyLoss()
    for ep in range(cfg.nn_epochs):
        model.train()
        idx = torch.randperm(ds['X_train'].shape[0], device=cfg.device)
        for s in range(0, ds['X_train'].shape[0], cfg.nn_batch_size):
            bi = idx[s:s+cfg.nn_batch_size]
            opt.zero_grad(); crit(model(ds['X_train'][bi]), ds['y_train'][bi]).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        tr = (model(ds['X_train']).argmax(1) == ds['y_train']).float().mean().item()
        te = (model(ds['X_test']).argmax(1) == ds['y_test']).float().mean().item()
    return tr, te, sum(p.numel() for p in model.parameters())


def train_tabular_baselines(ds, cfg):
    """Train Random Forest and XGBoost for tabular comparisons.
    Returns dict of {'rf': {...}, 'xgb': {...}} (xgb omitted if not installed)."""
    X_tr = ds['X_train'].cpu().numpy()
    y_tr = ds['y_train'].cpu().numpy()
    X_te = ds['X_test'].cpu().numpy()
    y_te = ds['y_test'].cpu().numpy()

    results = {}

    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=200, random_state=cfg.seed,
                                n_jobs=-1, min_samples_leaf=2)
    rf.fit(X_tr, y_tr)
    results['rf'] = {
        'train': rf.score(X_tr, y_tr),
        'test': rf.score(X_te, y_te),
        'n_estimators': 200,
    }

    try:
        from xgboost import XGBClassifier
        n_cls = len(set(y_tr.tolist()))
        xgb = XGBClassifier(n_estimators=200, max_depth=6,
                            random_state=cfg.seed,
                            eval_metric='mlogloss' if n_cls > 2 else 'logloss',
                            tree_method='hist', n_jobs=-1, verbosity=0)
        xgb.fit(X_tr, y_tr)
        results['xgb'] = {
            'train': xgb.score(X_tr, y_tr),
            'test': xgb.score(X_te, y_te),
            'n_estimators': 200,
        }
    except ImportError:
        print("  [skip] xgboost not installed (pip install xgboost)")

    return results


# ============================================================
# MAIN
# ============================================================

def run(cfg):
    print("=" * 70)
    print("  FMA EVOLUTION LAB v2.4 -- STACKED GP ENSEMBLE")
    print("  GP Members + Learned Meta-Combiner")
    print("=" * 70)

    if cfg.device == 'cuda' and not torch.cuda.is_available():
        cfg.device = 'cpu'
    if cfg.device == 'cuda':
        print(f"  GPU: {torch.cuda.get_device_name(0)}")

    torch.manual_seed(cfg.seed); random.seed(cfg.seed); np.random.seed(cfg.seed)
    set_active_ops(cfg.tier)
    print(f"  Active primitives ({cfg.tier}): {', '.join(ACTIVE_OP_KEYS)}")

    ds = load_dataset(cfg.tier, cfg.device, cfg)
    print(f"\n  Dataset: {ds['name']} ({ds['X_train'].shape[0]} train / "
          f"{ds['X_val'].shape[0]} val / {ds['X_test'].shape[0]} test)")

    # NN baseline
    print(f"\n{'-'*70}")
    print(f"  PHASE 0: NEURAL NETWORK BASELINE")
    print(f"{'-'*70}")
    nn_tr, nn_te, nn_p = train_baseline(ds, cfg)
    print(f"  NN: train={nn_tr:.1%}  test={nn_te:.1%}  ({nn_p:,} params)")

    # Tabular baselines (Tiers 3 and 4) -- where GPEs actually have to compete
    tab_baselines = {}
    if cfg.tier in (3, 4, 5):
        print(f"\n{'-'*70}")
        print(f"  PHASE 0b: TABULAR BASELINES (Random Forest + XGBoost)")
        print(f"{'-'*70}")
        tab_baselines = train_tabular_baselines(ds, cfg)
        if 'rf' in tab_baselines:
            r = tab_baselines['rf']
            print(f"  RF:  train={r['train']:.1%}  test={r['test']:.1%}")
        if 'xgb' in tab_baselines:
            x = tab_baselines['xgb']
            print(f"  XGB: train={x['train']:.1%}  test={x['test']:.1%}")

    # Evolve ensemble
    print(f"\n{'='*70}")
    print(f"  PHASE 1: EVOLVING {cfg.num_members} GP ENSEMBLE MEMBERS")
    if cfg.boost:
        print(f"  Diversity boosting ON (weight={cfg.boost_weight}x after member {cfg.boost_after})")
    print(f"{'='*70}\n")

    members = []
    evo_start = time.time()

    # Pre-compute per-sample class weights if enabled (inverse-prevalence)
    class_weights_per_sample = None
    if cfg.class_weight:
        counts = torch.bincount(ds['y_train'])
        freq = counts.float() / counts.sum()
        inv = 1.0 / (freq + 1e-8)
        inv = inv / inv.mean()
        class_weights_per_sample = inv[ds['y_train']]
        print(f"  Class-weighted fitness ON (inv-prevalence weights per class: "
              f"{[f'{v:.2f}' for v in inv.cpu().tolist()]})")

    print(f"  {'#':>3} {'Ind.Test':>9} {'Size':>6} {'MajVote':>9} {'Time':>6}")
    print(f"  {'-'*40}")

    for m in range(1, cfg.num_members + 1):
        t0 = time.time()

        # Sample weights: combine (optionally) class and boost weights
        weights = None
        if cfg.boost and m > cfg.boost_after and len(members) >= cfg.boost_after:
            with torch.no_grad():
                ens_preds = majority_vote(members, ds['X_train'], ds['num_classes'])
                wrong = (ens_preds != ds['y_train'])
                weights = torch.ones(ds['X_train'].shape[0], device=cfg.device)
                weights[wrong] = cfg.boost_weight

        if class_weights_per_sample is not None:
            weights = class_weights_per_sample.clone() if weights is None \
                      else weights * class_weights_per_sample

        if weights is not None:
            weights = weights / weights.mean()

        member = evolve_member(ds['X_train'], ds['y_train'], cfg,
                              weights=weights, member_num=m)
        members.append(member)

        ind_test = get_accuracy(member, ds['X_test'], ds['y_test'])
        maj_preds = majority_vote(members, ds['X_test'], ds['num_classes'])
        maj_test = (maj_preds == ds['y_test']).float().mean().item()
        elapsed = time.time() - t0

        boost_tag = " B" if weights is not None else "  "
        print(f"  {m:>3}{boost_tag}{ind_test:>7.1%} {member.total_size():>5}n"
              f"  {maj_test:>7.1%}  {elapsed:>5.0f}s")

    evo_time = time.time() - evo_start
    print(f"\n  Ensemble evolved in {evo_time:.0f}s")

    # Extract stacked features
    print(f"\n{'='*70}")
    print(f"  PHASE 2: STACKING -- TRAINING META-LEARNERS")
    print(f"{'='*70}")

    with torch.no_grad():
        stacked_train = extract_stacked_features(members, ds['X_train'])
        stacked_val = extract_stacked_features(members, ds['X_val'])
        stacked_test = extract_stacked_features(members, ds['X_test'])

    feat_dim = stacked_train.shape[1]
    print(f"\n  Stacked feature dimension: {feat_dim}"
          f" ({cfg.num_members} members x {ds['num_classes']} classes)")

    # 1) Majority vote (baseline combination)
    print(f"\n  --- Method 1: Majority Vote ---")
    with torch.no_grad():
        mv_train_preds = majority_vote(members, ds['X_train'], ds['num_classes'])
        mv_test_preds = majority_vote(members, ds['X_test'], ds['num_classes'])
        mv_train = (mv_train_preds == ds['y_train']).float().mean().item()
        mv_test = (mv_test_preds == ds['y_test']).float().mean().item()
    print(f"  train={mv_train:.1%}  test={mv_test:.1%}")

    # 2) Weighted vote (accuracy-proportional)
    print(f"\n  --- Method 2: Accuracy-Weighted Vote ---")
    wv_train, wv_test, wv_weights = weighted_vote_meta(
        stacked_train, ds['y_train'], stacked_test, ds['y_test'],
        cfg.num_members, ds['num_classes'])
    print(f"  train={wv_train:.1%}  test={wv_test:.1%}")

    # 3) Logistic regression
    print(f"\n  --- Method 3: Logistic Regression (sklearn) ---")
    lr_train, lr_test, lr_model, lr_scaler = train_logistic_meta(
        stacked_train, ds['y_train'], stacked_test, ds['y_test'])
    print(f"  train={lr_train:.1%}  test={lr_test:.1%}")

    # 4) MLP meta-learner (val-based early stopping)
    print(f"\n  --- Method 4: MLP Meta-Learner ({feat_dim}->{cfg.meta_hidden}->{cfg.meta_hidden//2}->{ds['num_classes']}) ---")
    mlp_train, mlp_val, mlp_test, mlp_model, mlp_mean, mlp_std, mlp_params = train_mlp_meta(
        stacked_train, ds['y_train'],
        stacked_val, ds['y_val'],
        stacked_test, ds['y_test'], cfg, cfg.device)
    print(f"  train={mlp_train:.1%}  val={mlp_val:.1%}  test={mlp_test:.1%}  ({mlp_params:,} meta-params)")

    # 5) XGBoost meta-learner on stacked GP features
    print(f"\n  --- Method 5: XGBoost Meta on Stacked GP Features ---")
    xgb_meta = train_xgb_meta(stacked_train, ds['y_train'],
                              stacked_val, ds['y_val'],
                              stacked_test, ds['y_test'], cfg)
    if xgb_meta:
        print(f"  train={xgb_meta['train']:.1%}  val={xgb_meta['val']:.1%}  test={xgb_meta['test']:.1%}"
              f"  ({xgb_meta['trees_used']} trees used)")
    else:
        print(f"  [skip] xgboost not installed")

    # 6) XGBoost meta-learner on concat(raw, stacked) -- does GP add signal beyond raw?
    print(f"\n  --- Method 6: XGBoost Meta on RAW+Stacked Features (augmented) ---")
    xgb_aug = train_xgb_augmented_meta(
        ds['X_train'], stacked_train, ds['y_train'],
        ds['X_val'], stacked_val, ds['y_val'],
        ds['X_test'], stacked_test, ds['y_test'], cfg)
    if xgb_aug:
        print(f"  train={xgb_aug['train']:.1%}  val={xgb_aug['val']:.1%}  test={xgb_aug['test']:.1%}"
              f"  ({xgb_aug['trees_used']} trees, {xgb_aug['input_dim']}-dim input)")
        print(f"  Feature importance split: raw={xgb_aug['raw_importance']:.1%}  "
              f"gp-stacked={xgb_aug['stacked_importance']:.1%}")
    else:
        print(f"  [skip] xgboost not installed")

    # Best-method selection includes the new meta-learners
    candidates = [('Majority Vote', mv_test), ('Weighted Vote', wv_test),
                  ('Logistic Reg', lr_test), ('MLP Meta', mlp_test)]
    if xgb_meta:
        candidates.append(('XGB-Meta', xgb_meta['test']))
    if xgb_aug:
        candidates.append(('XGB-Meta-Aug', xgb_aug['test']))
    best_method = max(candidates, key=lambda x: x[1])

    # ============================================================
    # FINAL REPORT
    # ============================================================

    total_gp_nodes = sum(m.total_size() for m in members)
    avg_ind_test = sum(get_accuracy(m, ds['X_test'], ds['y_test']) for m in members) / len(members)
    best_ind_test = max(get_accuracy(m, ds['X_test'], ds['y_test']) for m in members)

    print(f"\n{'='*70}")
    print(f"  FINAL RESULTS")
    print(f"{'='*70}")
    print(f"\n  {'METHOD':>30} {'TRAIN':>8} {'TEST':>8} {'GAP':>8} {'PARAMS/NODES':>14}")
    print(f"  {'-'*68}")
    print(f"  {'GP Best Individual':>30} {'':>8} {best_ind_test:>8.1%}")
    print(f"  {'GP Avg Individual':>30} {'':>8} {avg_ind_test:>8.1%}")
    print(f"  {'':>30} {'':>8} {'':>8} {'':>8} {'':>14}")
    print(f"  {'Majority Vote':>30} {mv_train:>8.1%} {mv_test:>8.1%}"
          f" {mv_train-mv_test:>+8.1%} {total_gp_nodes:>10} nodes")
    print(f"  {'Accuracy-Weighted Vote':>30} {wv_train:>8.1%} {wv_test:>8.1%}"
          f" {wv_train-wv_test:>+8.1%} {total_gp_nodes:>10} nodes")
    print(f"  {'Logistic Regression':>30} {lr_train:>8.1%} {lr_test:>8.1%}"
          f" {lr_train-lr_test:>+8.1%} {'':>10}")
    print(f"  {'MLP Meta-Learner':>30} {mlp_train:>8.1%} {mlp_test:>8.1%}"
          f" {mlp_train-mlp_test:>+8.1%} {mlp_params:>10} meta-p")
    if xgb_meta:
        print(f"  {'XGB Meta on GP features':>30} {xgb_meta['train']:>8.1%} {xgb_meta['test']:>8.1%}"
              f" {xgb_meta['train']-xgb_meta['test']:>+8.1%} {xgb_meta['trees_used']:>10} trees")
    if xgb_aug:
        print(f"  {'XGB Meta on RAW+GP (aug)':>30} {xgb_aug['train']:>8.1%} {xgb_aug['test']:>8.1%}"
              f" {xgb_aug['train']-xgb_aug['test']:>+8.1%} {xgb_aug['trees_used']:>10} trees")
    print(f"  {'':>30} {'':>8} {'':>8} {'':>8} {'':>14}")
    print(f"  {'Standard Neural Network':>30} {nn_tr:>8.1%} {nn_te:>8.1%}"
          f" {nn_tr-nn_te:>+8.1%} {nn_p:>10} params")
    # Tabular baselines (Tier 3)
    if 'rf' in tab_baselines:
        r = tab_baselines['rf']
        print(f"  {'Random Forest (200 trees)':>30} {r['train']:>8.1%} {r['test']:>8.1%}"
              f" {r['train']-r['test']:>+8.1%} {'200':>10} trees")
    if 'xgb' in tab_baselines:
        x = tab_baselines['xgb']
        print(f"  {'XGBoost (200 trees, d=6)':>30} {x['train']:>8.1%} {x['test']:>8.1%}"
              f" {x['train']-x['test']:>+8.1%} {'200':>10} trees")

    print(f"\n  Best stacking method: {best_method[0]} ({best_method[1]:.1%})")
    gap_to_nn = best_method[1] - nn_te
    if gap_to_nn >= 0:
        print(f"  --> STACKED GP MATCHES OR BEATS THE NEURAL NETWORK! ({gap_to_nn:+.1%})")
    else:
        print(f"  --> Gap to NN: {gap_to_nn:+.1%}")

    # Tabular-specific competitive framing
    if 'xgb' in tab_baselines:
        gap_to_xgb = best_method[1] - tab_baselines['xgb']['test']
        if gap_to_xgb >= 0:
            print(f"  --> STACKED GP BEATS XGBOOST! ({gap_to_xgb:+.1%})")
        else:
            print(f"  --> Gap to XGBoost: {gap_to_xgb:+.1%}")
    if 'rf' in tab_baselines:
        gap_to_rf = best_method[1] - tab_baselines['rf']['test']
        if gap_to_rf >= 0:
            print(f"  --> STACKED GP BEATS RANDOM FOREST! ({gap_to_rf:+.1%})")
        else:
            print(f"  --> Gap to Random Forest: {gap_to_rf:+.1%}")

    # The collaboration test: does GP as a preprocessing layer make XGBoost better?
    if xgb_meta and 'xgb' in tab_baselines:
        raw_xgb = tab_baselines['xgb']['test']
        print(f"\n  [COLLABORATION TEST -- does GP make XGBoost better?]")
        delta_m = xgb_meta['test'] - raw_xgb
        label = "  MAKES XGB BETTER" if delta_m > 0 else "  no help" if abs(delta_m) < 0.003 else "  HURTS XGB"
        print(f"    XGB on GP features only:   {xgb_meta['test']:.2%}  (vs raw XGB {raw_xgb:.2%}, delta {delta_m:+.2%}){label}")
        if xgb_aug:
            delta_a = xgb_aug['test'] - raw_xgb
            label = "  MAKES XGB BETTER" if delta_a > 0 else "  no help" if abs(delta_a) < 0.003 else "  HURTS XGB"
            print(f"    XGB on RAW+GP features:    {xgb_aug['test']:.2%}  (vs raw XGB {raw_xgb:.2%}, delta {delta_a:+.2%}){label}")
            print(f"    XGB attributes {xgb_aug['stacked_importance']:.1%} of importance to the GP features")

    # Architecture description
    print(f"\n  HYBRID ARCHITECTURE:")
    print(f"    Layer 1: {cfg.num_members} evolved GP programs ({total_gp_nodes} total FMA nodes)")
    print(f"    Layer 2: {best_method[0]}"
          f" ({mlp_params if 'MLP' in best_method[0] else 'no learnable'} params)")
    print(f"    Total: {total_gp_nodes} GP nodes"
          f" + {mlp_params if 'MLP' in best_method[0] else 0} meta-params")
    print(f"    vs NN:  {nn_p:,} dense params")

    # Per-class for best method
    print(f"\n  PER-CLASS ACCURACY (best stacking vs NN):")
    if 'MLP' in best_method[0]:
        with torch.no_grad():
            X_te_norm = (stacked_test - mlp_mean) / mlp_std
            best_preds = mlp_model(X_te_norm).argmax(1)
    elif 'Logistic' in best_method[0]:
        X_te_s = lr_scaler.transform(stacked_test.cpu().numpy())
        best_preds = torch.tensor(lr_model.predict(X_te_s), device=cfg.device)
    elif 'XGB-Meta-Aug' in best_method[0] and xgb_aug:
        best_preds = torch.tensor(xgb_aug['test_preds'], device=cfg.device)
    elif 'XGB-Meta' in best_method[0] and xgb_meta:
        best_preds = torch.tensor(xgb_meta['test_preds'], device=cfg.device)
    else:
        best_preds = majority_vote(members, ds['X_test'], ds['num_classes'])

    # NN preds for comparison
    nn_model = BaselineNN(ds['input_size'], cfg.nn_hidden, ds['num_classes']).to(cfg.device)
    # Retrain quickly for per-class comparison
    # (we'll use the baseline numbers instead)

    print(f"  {'Class':>7} {'Stacked':>9} {'Samples':>9}")
    print(f"  {'-'*27}")
    for c in range(ds['num_classes']):
        mask = ds['y_test'] == c
        if mask.sum() > 0:
            cls_acc = (best_preds[mask] == c).float().mean().item()
            print(f"  {c:>7} {cls_acc:>9.1%} {mask.sum().item():>9}")

    # Save (strip non-serializable numpy arrays, coerce numpy scalars)
    def _clean(d):
        if not isinstance(d, dict): return d
        out = {}
        for k, v in d.items():
            if k == 'test_preds': continue
            if hasattr(v, 'item') and not isinstance(v, (list, dict, str)):
                v = v.item()
            out[k] = v
        return out

    results = {
        'dataset': ds['name'],
        'num_members': cfg.num_members,
        'boost': cfg.boost,
        'majority_vote': {'train': mv_train, 'test': mv_test},
        'weighted_vote': {'train': wv_train, 'test': wv_test},
        'logistic_reg': {'train': lr_train, 'test': lr_test},
        'mlp_meta': {'train': mlp_train, 'test': mlp_test, 'params': mlp_params},
        'xgb_meta': _clean(xgb_meta),
        'xgb_meta_augmented': _clean(xgb_aug),
        'nn_baseline': {'train': nn_tr, 'test': nn_te, 'params': nn_p},
        'tabular_baselines': tab_baselines,
        'gp_total_nodes': total_gp_nodes,
        'best_method': best_method[0],
        'best_test': best_method[1],
        'time_s': time.time() - evo_start,
    }
    fname = f'stacked_results_{ds["name"]}_{int(time.time())}.json'
    with open(fname, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n  Results saved: {fname}")
    print(f"{'='*70}")
    return results


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='FMA Stacked GP Ensemble')
    parser.add_argument('--tier', type=int, default=1, choices=[1, 2, 3, 4, 5])
    parser.add_argument('--members', type=int, default=25)
    parser.add_argument('--boost', action='store_true', default=True)
    parser.add_argument('--no-boost', action='store_false', dest='boost')
    parser.add_argument('--pop', type=int, default=200)
    parser.add_argument('--gens', type=int, default=80)
    parser.add_argument('--depth', type=int, default=8)
    parser.add_argument('--meta-hidden', type=int, default=64)
    parser.add_argument('--meta-epochs', type=int, default=200)
    parser.add_argument('--boost-weight', type=float, default=3.0)
    parser.add_argument('--nn-hidden', type=int, default=128)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--cpu', action='store_true')
    parser.add_argument('--class-weight', action='store_true',
                        help='Enable inverse-prevalence class weighting in GP fitness')
    parser.add_argument('--covtype-subsample', type=int, default=50000,
                        help='Subsample size for Tier 4 covertype (0 = full 581k)')
    args = parser.parse_args()

    cfg = Config(
        tier=args.tier, num_members=args.members, boost=args.boost,
        pop_size=args.pop, gens=args.gens, max_depth=args.depth,
        meta_hidden=args.meta_hidden, meta_epochs=args.meta_epochs,
        boost_weight=args.boost_weight, nn_hidden=args.nn_hidden,
        class_weight=args.class_weight,
        covtype_subsample=args.covtype_subsample,
        seed=args.seed, device='cpu' if args.cpu else 'cuda',
    )

    est = cfg.num_members * 2.5
    print(f"\n  Stacked Ensemble: {cfg.num_members} GP members + meta-learner")
    print(f"  Estimated: ~{est:.0f} min for GP evolution + seconds for meta-training\n")
    run(cfg)


In [ ]:
%%writefile jit_kernel.py
"""
Fused GP+LR kernel — the headline engineering.

The single-sample path:
    1. Walk through each tree's bytecode using a small fixed-size stack
    2. Each tree produces one score; concatenate into stacked[]
    3. Multiply: output_logits = LR_W @ stacked + LR_b
    4. argmax -> predicted class

All in one @njit function, no Python in between.
"""
from __future__ import annotations

import numpy as np
from numba import njit

# Op codes (must match gp_core.py)
OP_FEAT  = 0
OP_CONST = 1
OP_ADD   = 2
OP_SUB   = 3
OP_MUL   = 4
OP_MAX   = 5
OP_MIN   = 6
OP_LT    = 7
OP_CLAMP = 8

# Stack size: deepest tree we'll ever evaluate.
# A binary tree of depth D needs at most D+1 stack slots in post-order.
# With max_depth=4 we need ~5 slots. Use 32 to be very safe.
STACK_SIZE = 32


# ============================================================
# Numpy reference implementation (for correctness checking)
# ============================================================
def numpy_gpe_lr_predict(X, instrs, operands, tree_starts, n_members, n_classes, lr_W, lr_b):
    """Reference pure-numpy implementation. Slow but obviously correct.
    X: (N, n_features) float32
    Returns: (N,) int64 predictions
    """
    N = X.shape[0]
    n_trees = n_members * n_classes
    stacked = np.zeros((N, n_trees), dtype=np.float32)

    for tree_idx in range(n_trees):
        start = tree_starts[tree_idx]
        end = tree_starts[tree_idx + 1]
        # Evaluate this tree's bytecode for all N samples
        stack = np.zeros((STACK_SIZE, N), dtype=np.float32)
        sp = 0
        for pc in range(start, end):
            op = instrs[pc]
            operand = operands[pc]
            if op == OP_FEAT:
                stack[sp] = X[:, int(operand)]
                sp += 1
            elif op == OP_CONST:
                stack[sp] = operand
                sp += 1
            elif op == OP_CLAMP:
                stack[sp-1] = np.clip(stack[sp-1], -1.0, 1.0)
            else:
                # Binary op: pop b, pop a, push f(a, b)
                a = stack[sp-2]
                b = stack[sp-1]
                if   op == OP_ADD: r = a + b
                elif op == OP_SUB: r = a - b
                elif op == OP_MUL: r = np.clip(a * b, -1e6, 1e6)
                elif op == OP_MAX: r = np.maximum(a, b)
                elif op == OP_MIN: r = np.minimum(a, b)
                elif op == OP_LT:  r = (a < b).astype(np.float32)
                else: raise ValueError(f"unknown op {op}")
                stack[sp-2] = r
                sp -= 1
        stacked[:, tree_idx] = stack[0]

    # LR layer: scores = stacked @ W.T + b ; argmax
    logits = stacked @ lr_W.T + lr_b
    return np.argmax(logits, axis=1)


# ============================================================
# Numba-JITed batch kernel — fused GP + LR
# ============================================================
@njit(cache=True, fastmath=True, boundscheck=False)
def jit_gpe_lr_predict_batch(X, instrs, operands, tree_starts, n_members, n_classes, lr_W, lr_b):
    """Single JIT call evaluating the entire pipeline for a batch of samples.

    Args:
        X: (N, n_features) float32
        instrs: (total_nodes,) int32
        operands: (total_nodes,) float32
        tree_starts: (n_members * n_classes + 1,) int32
        n_members: int
        n_classes: int
        lr_W: (n_lr_classes, n_members * n_classes) float32 -- LR weight matrix
        lr_b: (n_lr_classes,) float32

    Returns:
        preds: (N,) int64 -- predicted classes
    """
    N = X.shape[0]
    n_trees = n_members * n_classes
    n_lr_out = lr_W.shape[0]

    # We process each sample independently. For batch=1 this is the right structure
    # (avoids large allocs). For larger batches we still do per-sample loop because
    # numba's tree-walk is sequential; we get vectorization at the LR step inside the loop.
    preds = np.zeros(N, dtype=np.int64)
    stack = np.zeros(STACK_SIZE, dtype=np.float32)
    stacked = np.zeros(n_trees, dtype=np.float32)
    logits = np.zeros(n_lr_out, dtype=np.float32)

    for i in range(N):
        # ---- GP layer for sample i ----
        for tree_idx in range(n_trees):
            start = tree_starts[tree_idx]
            end = tree_starts[tree_idx + 1]
            sp = 0
            for pc in range(start, end):
                op = instrs[pc]
                operand = operands[pc]
                if op == OP_FEAT:
                    stack[sp] = X[i, np.int32(operand)]
                    sp += 1
                elif op == OP_CONST:
                    stack[sp] = operand
                    sp += 1
                elif op == OP_CLAMP:
                    v = stack[sp-1]
                    if v > 1.0: v = 1.0
                    elif v < -1.0: v = -1.0
                    stack[sp-1] = v
                else:
                    a = stack[sp-2]
                    b = stack[sp-1]
                    if op == OP_ADD:
                        stack[sp-2] = a + b
                    elif op == OP_SUB:
                        stack[sp-2] = a - b
                    elif op == OP_MUL:
                        v = a * b
                        if v > 1e6: v = 1e6
                        elif v < -1e6: v = -1e6
                        stack[sp-2] = v
                    elif op == OP_MAX:
                        stack[sp-2] = a if a > b else b
                    elif op == OP_MIN:
                        stack[sp-2] = a if a < b else b
                    elif op == OP_LT:
                        stack[sp-2] = 1.0 if a < b else 0.0
                    sp -= 1
            stacked[tree_idx] = stack[0]

        # ---- LR layer for sample i ----
        for c in range(n_lr_out):
            s = lr_b[c]
            for k in range(n_trees):
                s += lr_W[c, k] * stacked[k]
            logits[c] = s

        # ---- argmax ----
        best_c = 0
        best_v = logits[0]
        for c in range(1, n_lr_out):
            if logits[c] > best_v:
                best_v = logits[c]
                best_c = c
        preds[i] = best_c

    return preds


@njit(cache=True, fastmath=True, boundscheck=False)
def jit_gpe_score_batch(X, instrs, operands, tree_starts, n_members, n_classes):
    """JIT'd GP layer ALONE -- emits stacked features (no LR).

    Used for: training the LR meta on top of the JIT'd GP, and for
    head-to-head comparison with sklearn LR (where the GP is JIT but LR is not).
    """
    N = X.shape[0]
    n_trees = n_members * n_classes
    stacked = np.zeros((N, n_trees), dtype=np.float32)
    stack = np.zeros(STACK_SIZE, dtype=np.float32)

    for i in range(N):
        for tree_idx in range(n_trees):
            start = tree_starts[tree_idx]
            end = tree_starts[tree_idx + 1]
            sp = 0
            for pc in range(start, end):
                op = instrs[pc]
                operand = operands[pc]
                if op == OP_FEAT:
                    stack[sp] = X[i, np.int32(operand)]
                    sp += 1
                elif op == OP_CONST:
                    stack[sp] = operand
                    sp += 1
                elif op == OP_CLAMP:
                    v = stack[sp-1]
                    if v > 1.0: v = 1.0
                    elif v < -1.0: v = -1.0
                    stack[sp-1] = v
                else:
                    a = stack[sp-2]
                    b = stack[sp-1]
                    if op == OP_ADD:
                        stack[sp-2] = a + b
                    elif op == OP_SUB:
                        stack[sp-2] = a - b
                    elif op == OP_MUL:
                        v = a * b
                        if v > 1e6: v = 1e6
                        elif v < -1e6: v = -1e6
                        stack[sp-2] = v
                    elif op == OP_MAX:
                        stack[sp-2] = a if a > b else b
                    elif op == OP_MIN:
                        stack[sp-2] = a if a < b else b
                    elif op == OP_LT:
                        stack[sp-2] = 1.0 if a < b else 0.0
                    sp -= 1
            stacked[i, tree_idx] = stack[0]

    return stacked


In [ ]:
%%writefile bridge.py
"""
Bridge: gp_ensemble.py GPNode/GPIndividual  ->  JIT kernel bytecode.

gp_ensemble.py's GPNode uses string ops: ntype in {'input','const','op'},
value is the op name 'fma'/'add'/'mul'/etc.

Our JIT kernel needs int opcodes (OP_FEAT, OP_CONST, OP_ADD, etc.).
This module is the conversion.

NOTE: gp_ensemble.py supports more ops than our JIT kernel:
  - 'fma'  (3-arg fused multiply-add): we LOWER this to mul+add
  - 'neg', 'abs': we LOWER neg to (0 - x), abs to (max(x, -x))
  - 'add', 'mul', 'sub', 'max', 'min', 'lt', 'clamp': direct mapping

This is important because gp_ensemble.py's set_active_ops(tier=3) for Adult
uses ['add','mul','sub','max','min','clamp','lt'] which all map directly.
'fma' is only used on tiers 1/2 (image-like data).

We always lower defensively though, so the JIT kernel sees only its 7 ops.
"""

# Op codes (must match jit_kernel.py)
OP_FEAT  = 0
OP_CONST = 1
OP_ADD   = 2
OP_SUB   = 3
OP_MUL   = 4
OP_MAX   = 5
OP_MIN   = 6
OP_LT    = 7
OP_CLAMP = 8


def flatten_gpnode(node, instrs, operands):
    """Recursively flatten a gp_ensemble.GPNode -> append to instrs/operands lists.

    Post-order traversal: children pushed before the op that consumes them.
    """
    if node.ntype == 'input':
        instrs.append(OP_FEAT)
        operands.append(float(node.value))
        return
    if node.ntype == 'const':
        instrs.append(OP_CONST)
        operands.append(float(node.value))
        return

    # ntype == 'op' -- need to look at node.value (the op name)
    op_name = node.value

    if op_name == 'fma':
        # 3-arg: a*b + c. Lower to: mul(a,b), add(result, c)
        # Walk children in order
        flatten_gpnode(node.children[0], instrs, operands)
        flatten_gpnode(node.children[1], instrs, operands)
        instrs.append(OP_MUL); operands.append(0.0)
        flatten_gpnode(node.children[2], instrs, operands)
        instrs.append(OP_ADD); operands.append(0.0)
        return

    if op_name == 'neg':
        # -x  =>  0 - x. Push 0 first, then x, then SUB.
        instrs.append(OP_CONST); operands.append(0.0)
        flatten_gpnode(node.children[0], instrs, operands)
        instrs.append(OP_SUB); operands.append(0.0)
        return

    if op_name == 'abs':
        # |x| = max(x, -x). Compute x once, then push (0-x), then MAX.
        # Stack-machine note: we need x on the stack twice. We don't have a DUP op.
        # Trick: emit the subtree twice (once for the max arg, once inside the negation).
        # This produces correct results but increases bytecode size by ~one subtree.
        flatten_gpnode(node.children[0], instrs, operands)         # x
        # Now compute -x: 0 - x
        instrs.append(OP_CONST); operands.append(0.0)              # 0
        flatten_gpnode(node.children[0], instrs, operands)         # x (again)
        instrs.append(OP_SUB); operands.append(0.0)                # 0 - x = -x
        instrs.append(OP_MAX); operands.append(0.0)                # max(x, -x) = |x|
        return

    # Two-arg ops: walk children, then emit the op.
    # 'add', 'sub', 'mul', 'max', 'min', 'lt'
    if op_name in ('add', 'sub', 'mul', 'max', 'min', 'lt'):
        flatten_gpnode(node.children[0], instrs, operands)
        flatten_gpnode(node.children[1], instrs, operands)
        op_map = {'add': OP_ADD, 'sub': OP_SUB, 'mul': OP_MUL,
                  'max': OP_MAX, 'min': OP_MIN, 'lt': OP_LT}
        instrs.append(op_map[op_name])
        operands.append(0.0)
        return

    if op_name == 'clamp':
        # gp_ensemble's clamp is 1-arg, clamps to [0, 1]
        # Our JIT kernel's OP_CLAMP clamps to [-1, 1]
        # These are different! Need to be honest about this.
        # For now: use OP_CLAMP. The accuracy will be off by a constant
        # transformation that LR can adjust for. But we should test this.
        # SAFER LOWERING: emit max(min(x, 1), 0)
        #   compute x, push const 1, MIN, push const 0, MAX
        flatten_gpnode(node.children[0], instrs, operands)         # x
        instrs.append(OP_CONST); operands.append(1.0)              # 1
        instrs.append(OP_MIN); operands.append(0.0)                # min(x, 1)
        instrs.append(OP_CONST); operands.append(0.0)              # 0
        instrs.append(OP_MAX); operands.append(0.0)                # max(min(x, 1), 0) = clamp(x, 0, 1)
        return

    raise ValueError(f"Unsupported op: {op_name!r}")


def pack_ensemble_from_gp_ensemble(members):
    """Pack a list of gp_ensemble.GPIndividual into JIT-kernel arrays.

    Returns dict with: n_members, n_classes, instrs, operands, tree_starts, total_nodes.
    """
    import numpy as np
    n_members = len(members)
    n_classes = len(members[0].trees)

    all_instrs = []
    all_operands = []
    tree_starts = []

    for m in members:
        for c in range(n_classes):
            tree_starts.append(len(all_instrs))
            flatten_gpnode(m.trees[c], all_instrs, all_operands)
    tree_starts.append(len(all_instrs))  # sentinel

    return {
        'n_members': n_members,
        'n_classes': n_classes,
        'instrs': np.array(all_instrs, dtype=np.int32),
        'operands': np.array(all_operands, dtype=np.float32),
        'tree_starts': np.array(tree_starts, dtype=np.int32),
        'total_nodes': len(all_instrs),
    }


In [ ]:
%%writefile lightning_aot.py
"""
Lightning AOT C99 emitter.

Takes a packed GP+LR ensemble (the bytecode produced by bridge.pack_ensemble_from_gp_ensemble
plus a folded LR head) and emits a self-contained C99 source file with one
straight-line predict() function per sample. No interpreter loop, no opcode dispatch,
no runtime stack: every GP op becomes a single C statement against named temporaries,
the LR layer is a baked-in dot product with constants as immediates.

Compile with:
    gcc -O3 -march=native -fno-plt -shared -fPIC out.c -o out.so

API of the emitted library:
    int32_t predict_one(const float* X);                 // single-sample, returns class
    void    predict_batch(const float* X, int32_t N,
                          int32_t n_features, int32_t* out);   // batch loop

The emitter performs constant folding and the standard algebraic identities GP
evolves a lot of (x+0, x*1, x*0, 0-x, etc). The output is small, branch-light,
and friendly to the autovectorizer.
"""

from __future__ import annotations
import ctypes
import os
import subprocess
import tempfile
from dataclasses import dataclass
from typing import List, Optional

import numpy as np

OP_FEAT  = 0
OP_CONST = 1
OP_ADD   = 2
OP_SUB   = 3
OP_MUL   = 4
OP_MAX   = 5
OP_MIN   = 6
OP_LT    = 7
OP_CLAMP = 8


# ---------------------------------------------------------------------------
# Symbolic stack-machine walker -> C expression strings
# ---------------------------------------------------------------------------
@dataclass
class Expr:
    """A C-expression with optional compile-time constant value.

    If `const_val` is not None the expression is a literal and we can fold.
    Otherwise `text` is the C expression text.
    """
    text: str
    const_val: Optional[float] = None

    @staticmethod
    def lit(v: float) -> "Expr":
        return Expr(_fmt_const(v), const_val=float(v))

    @staticmethod
    def var(name: str) -> "Expr":
        return Expr(name, const_val=None)


def _fmt_const(v: float) -> str:
    # Emit as float32 literal. Inf / NaN handled defensively.
    if np.isnan(v):
        return "(0.0f/0.0f)"
    if np.isposinf(v):
        return "(1.0f/0.0f)"
    if np.isneginf(v):
        return "(-1.0f/0.0f)"
    # %.9g preserves float32 round-trip. Ensure a '.' is present so the 'f'
    # suffix is parsed as a float literal (otherwise '0f' is an invalid
    # integer suffix).
    s = f"{v:.9g}"
    if "." not in s and "e" not in s and "E" not in s:
        s += ".0"
    return s + "f"


def _emit_tree(instrs: np.ndarray, operands: np.ndarray,
               start: int, end: int,
               tmp_counter: List[int],
               body: List[str]) -> Expr:
    """Walk one tree's bytecode, emit C statements into `body`,
    return the Expr for the tree's output."""
    stack: List[Expr] = []

    def push(e: Expr):
        stack.append(e)

    def materialize(e: Expr) -> Expr:
        """If e is a complex expression we want to bind it to a temp before
        reusing it. For straight stack-machine code we mostly DON'T need to,
        because each value is consumed at most once. But a binary op like
        a+b produces a result that's pushed and then consumed by the next
        op -- the C compiler will CSE / register-allocate fine if we just
        inline as text. We materialize selectively to keep generated lines short."""
        return e

    def fold_binop(op: str, a: Expr, b: Expr,
                    py_fn) -> Expr:
        if a.const_val is not None and b.const_val is not None:
            return Expr.lit(py_fn(a.const_val, b.const_val))
        return None

    def alg_simplify_add(a: Expr, b: Expr) -> Optional[Expr]:
        if a.const_val == 0.0: return b
        if b.const_val == 0.0: return a
        return None

    def alg_simplify_sub(a: Expr, b: Expr) -> Optional[Expr]:
        if b.const_val == 0.0: return a
        return None

    def alg_simplify_mul(a: Expr, b: Expr) -> Optional[Expr]:
        if a.const_val == 0.0 or b.const_val == 0.0:
            return Expr.lit(0.0)
        if a.const_val == 1.0: return b
        if b.const_val == 1.0: return a
        if a.const_val == -1.0:
            return Expr(f"(-({b.text}))")
        if b.const_val == -1.0:
            return Expr(f"(-({a.text}))")
        return None

    def emit_binop(c_op: str, a: Expr, b: Expr) -> Expr:
        # Expression text. Parenthesize for safety.
        return Expr(f"({a.text} {c_op} {b.text})")

    def new_tmp() -> str:
        tmp_counter[0] += 1
        return f"t{tmp_counter[0]}"

    def maybe_bind(e: Expr) -> Expr:
        """Bind big text expressions to a temp; keep small ones inline.
        This is a crude heuristic: if the text is longer than ~80 chars,
        bind. Keeps generated lines readable AND helps the compiler avoid
        re-evaluating common sub-expressions when we deliberately CSE."""
        if e.const_val is not None:
            return e
        if len(e.text) > 80:
            name = new_tmp()
            body.append(f"    const float {name} = {e.text};")
            return Expr.var(name)
        return e

    pc = start
    while pc < end:
        op = int(instrs[pc])
        operand = float(operands[pc])
        if op == OP_FEAT:
            idx = int(operand)
            push(Expr.var(f"X[{idx}]"))
        elif op == OP_CONST:
            push(Expr.lit(operand))
        elif op == OP_ADD:
            b = stack.pop(); a = stack.pop()
            folded = fold_binop("+", a, b, lambda x, y: x + y)
            if folded is None:
                folded = alg_simplify_add(a, b) or emit_binop("+", a, b)
            push(maybe_bind(folded))
        elif op == OP_SUB:
            b = stack.pop(); a = stack.pop()
            folded = fold_binop("-", a, b, lambda x, y: x - y)
            if folded is None:
                folded = alg_simplify_sub(a, b) or emit_binop("-", a, b)
            push(maybe_bind(folded))
        elif op == OP_MUL:
            b = stack.pop(); a = stack.pop()
            # Match the JIT: clamp product to +/-1e6.
            folded = fold_binop("*", a, b,
                                lambda x, y: max(-1e6, min(1e6, x * y)))
            if folded is None:
                simp = alg_simplify_mul(a, b)
                if simp is not None:
                    folded = simp
                else:
                    name = new_tmp()
                    body.append(
                        f"    const float {name} = "
                        f"clamp_mul({a.text}, {b.text});"
                    )
                    folded = Expr.var(name)
            push(folded)
        elif op == OP_MAX:
            b = stack.pop(); a = stack.pop()
            folded = fold_binop("max", a, b, lambda x, y: max(x, y))
            if folded is None:
                folded = Expr(f"fmaxf({a.text}, {b.text})")
            push(maybe_bind(folded))
        elif op == OP_MIN:
            b = stack.pop(); a = stack.pop()
            folded = fold_binop("min", a, b, lambda x, y: min(x, y))
            if folded is None:
                folded = Expr(f"fminf({a.text}, {b.text})")
            push(maybe_bind(folded))
        elif op == OP_LT:
            b = stack.pop(); a = stack.pop()
            folded = fold_binop("<", a, b, lambda x, y: 1.0 if x < y else 0.0)
            if folded is None:
                folded = Expr(f"(({a.text}) < ({b.text}) ? 1.0f : 0.0f)")
            push(maybe_bind(folded))
        elif op == OP_CLAMP:
            a = stack.pop()
            if a.const_val is not None:
                push(Expr.lit(max(-1.0, min(1.0, a.const_val))))
            else:
                push(maybe_bind(Expr(f"fmaxf(-1.0f, fminf(1.0f, {a.text}))")))
        else:
            raise ValueError(f"unknown opcode {op} at pc={pc}")
        pc += 1

    if len(stack) != 1:
        raise RuntimeError(
            f"stack imbalance after tree: {len(stack)} items remain "
            f"(start={start}, end={end})"
        )
    return stack[0]


# ---------------------------------------------------------------------------
# Public API: emit C source for a packed ensemble + folded LR head
# ---------------------------------------------------------------------------
def emit_c_source(pack: dict,
                  lr_W: np.ndarray, lr_b: np.ndarray,
                  n_features: int,
                  func_prefix: str = "predict") -> str:
    """Emit a C99 source file string.

    pack: dict with keys 'instrs' (int32), 'operands' (float32),
          'tree_starts' (int32), 'n_members', 'n_classes'.
          n_trees = n_members * n_classes.
    lr_W: (n_lr_out, n_trees) float32  -- LR weight matrix (already scaler-folded)
    lr_b: (n_lr_out,) float32          -- LR bias (already scaler-folded)
    n_features: int -- feature width of X
    """
    instrs = np.asarray(pack["instrs"], dtype=np.int32)
    operands = np.asarray(pack["operands"], dtype=np.float32)
    tree_starts = np.asarray(pack["tree_starts"], dtype=np.int32)
    n_members = int(pack["n_members"])
    n_classes = int(pack["n_classes"])
    n_trees = n_members * n_classes
    assert tree_starts.shape[0] == n_trees + 1, "tree_starts must include sentinel"
    assert lr_W.ndim == 2 and lr_W.shape[1] == n_trees
    n_lr_out = lr_W.shape[0]
    assert lr_b.shape == (n_lr_out,)

    lines: List[str] = []
    L = lines.append

    L("/* Lightning AOT-emitted predictor. Do not edit by hand. */")
    L("#include <math.h>")
    L("#include <stddef.h>")
    L("#include <stdint.h>")
    L("")
    L("static inline float clamp_mul(float a, float b) {")
    L("    float v = a * b;")
    L("    if (v > 1e6f) v = 1e6f;")
    L("    else if (v < -1e6f) v = -1e6f;")
    L("    return v;")
    L("}")
    L("")
    L(f"int32_t {func_prefix}_one(const float* X) {{")

    # ---- per-tree codegen ----
    tree_outs: List[str] = []
    tmp_counter = [0]
    for t in range(n_trees):
        s = int(tree_starts[t]); e = int(tree_starts[t + 1])
        body: List[str] = []
        body.append(f"    /* tree {t} : pc {s}..{e} */")
        out_expr = _emit_tree(instrs, operands, s, e, tmp_counter, body)
        # Always bind the tree output to a named slot so the LR section is uniform.
        tree_var = f"s{t}"
        body.append(f"    const float {tree_var} = {out_expr.text};")
        tree_outs.append(tree_var)
        for ln in body:
            L(ln)
        L("")

    # ---- LR head: dot products with baked-in weights ----
    L("    /* LR head (folded scaler+weights, immediates) */")
    logit_terms = []
    for c in range(n_lr_out):
        terms = [_fmt_const(float(lr_b[c]))]
        for k in range(n_trees):
            w = float(lr_W[c, k])
            if w == 0.0:
                continue
            terms.append(f"{_fmt_const(w)}*{tree_outs[k]}")
        # Emit one logit per output. The compiler will fmadd / vectorize.
        L(f"    const float logit{c} = " + " + ".join(terms) + ";")
        logit_terms.append(f"logit{c}")

    # ---- argmax ----
    L("")
    L("    /* argmax */")
    if n_lr_out == 1:
        # Special case: single-output -> threshold at 0
        L("    return logit0 > 0.0f ? 1 : 0;")
    elif n_lr_out == 2:
        # Common binary case
        L("    return logit1 > logit0 ? 1 : 0;")
    else:
        L("    int32_t best = 0;")
        L("    float bestv = logit0;")
        for c in range(1, n_lr_out):
            L(f"    if (logit{c} > bestv) {{ bestv = logit{c}; best = {c}; }}")
        L("    return best;")
    L("}")
    L("")

    # Batch wrapper.
    L(f"void {func_prefix}_batch(const float* X, int32_t N, int32_t n_features, int32_t* out) {{")
    L("    for (int32_t i = 0; i < N; ++i) {")
    L(f"        out[i] = {func_prefix}_one(X + (size_t)i * (size_t)n_features);")
    L("    }")
    L("}")
    L("")

    # C-side timing harness: runs N inferences on the same sample, measured
    # entirely in C with clock_gettime. Returns mean nanoseconds per call.
    # Eliminates ALL Python/ctypes overhead from the measurement.
    L("#include <time.h>")
    L("")
    L(f"double {func_prefix}_time_ns(const float* X, int32_t n_warmup, int32_t n_iters) {{")
    L("    /* Warm icache / branch predictors. Result XOR'd into a sink so the")
    L("       compiler cannot dead-code eliminate the call.            */")
    L("    volatile int32_t sink = 0;")
    L("    for (int32_t i = 0; i < n_warmup; ++i) {")
    L(f"        sink ^= {func_prefix}_one(X);")
    L("    }")
    L("    struct timespec t0, t1;")
    L("    clock_gettime(CLOCK_MONOTONIC, &t0);")
    L("    for (int32_t i = 0; i < n_iters; ++i) {")
    L(f"        sink ^= {func_prefix}_one(X);")
    L("    }")
    L("    clock_gettime(CLOCK_MONOTONIC, &t1);")
    L("    (void) sink;")
    L("    double dt_ns = (double)(t1.tv_sec - t0.tv_sec) * 1e9 +")
    L("                    (double)(t1.tv_nsec - t0.tv_nsec);")
    L("    return dt_ns / (double)n_iters;")
    L("}")
    L("")

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Compile + load
# ---------------------------------------------------------------------------
def compile_c(src: str, *, out_dir: Optional[str] = None,
              cflags: Optional[List[str]] = None) -> str:
    """Compile a C source string into a shared object. Returns path to .so."""
    out_dir = out_dir or tempfile.mkdtemp(prefix="lightning_aot_")
    src_path = os.path.join(out_dir, "predict.c")
    so_path = os.path.join(out_dir, "predict.so")
    with open(src_path, "w") as f:
        f.write(src)
    cflags = cflags or [
        "-O3", "-march=native", "-fno-plt",
        "-fno-math-errno", "-fno-trapping-math",
        "-shared", "-fPIC",
    ]
    cmd = ["gcc", *cflags, src_path, "-o", so_path, "-lm"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"gcc failed:\n  cmd: {' '.join(cmd)}\n"
            f"  stderr:\n{res.stderr}\n"
            f"  source written at: {src_path}"
        )
    return so_path


class AOTPredictor:
    """ctypes wrapper around the compiled .so."""

    def __init__(self, so_path: str, n_features: int, func_prefix: str = "predict"):
        self.so_path = so_path
        self.n_features = n_features
        self._lib = ctypes.CDLL(so_path)
        self._one = getattr(self._lib, f"{func_prefix}_one")
        self._one.argtypes = [ctypes.POINTER(ctypes.c_float)]
        self._one.restype = ctypes.c_int32
        self._batch = getattr(self._lib, f"{func_prefix}_batch")
        self._batch.argtypes = [
            ctypes.POINTER(ctypes.c_float), ctypes.c_int32,
            ctypes.c_int32, ctypes.POINTER(ctypes.c_int32),
        ]
        self._batch.restype = None
        self._time_ns = getattr(self._lib, f"{func_prefix}_time_ns")
        self._time_ns.argtypes = [
            ctypes.POINTER(ctypes.c_float), ctypes.c_int32, ctypes.c_int32,
        ]
        self._time_ns.restype = ctypes.c_double

    def predict_one(self, x: np.ndarray) -> int:
        x = np.ascontiguousarray(x, dtype=np.float32).reshape(-1)
        assert x.shape[0] == self.n_features, f"expected {self.n_features} feats"
        return int(self._one(x.ctypes.data_as(ctypes.POINTER(ctypes.c_float))))

    def time_ns_per_call(self, x: np.ndarray,
                          n_warmup: int = 10000, n_iters: int = 1_000_000) -> float:
        """Time predict_one entirely in C. Returns ns/call. No Python overhead."""
        x = np.ascontiguousarray(x, dtype=np.float32).reshape(-1)
        return float(self._time_ns(
            x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            n_warmup, n_iters,
        ))

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.ascontiguousarray(X, dtype=np.float32)
        assert X.ndim == 2 and X.shape[1] == self.n_features
        N = X.shape[0]
        out = np.empty(N, dtype=np.int32)
        self._batch(
            X.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            N, self.n_features,
            out.ctypes.data_as(ctypes.POINTER(ctypes.c_int32)),
        )
        return out


def build_aot(pack: dict, lr_W: np.ndarray, lr_b: np.ndarray,
              n_features: int, *, out_dir: Optional[str] = None) -> AOTPredictor:
    """Convenience: emit -> compile -> load."""
    src = emit_c_source(pack, lr_W, lr_b, n_features)
    so_path = compile_c(src, out_dir=out_dir)
    return AOTPredictor(so_path, n_features=n_features)


In [ ]:
import os, sys
for f in ['gp_ensemble.py', 'jit_kernel.py', 'bridge.py', 'lightning_aot.py']:
    print(f'  {f:<22}  {os.path.getsize(f):>7} bytes' if os.path.exists(f) else f'  {f:<22}  MISSING')
for m in ['gp_ensemble', 'jit_kernel', 'bridge', 'lightning_aot']:
    if m in sys.modules: del sys.modules[m]


## 3. Load training data

Demo: UCI Pima-Diabetes. **Replace this cell with your own dataset loader.**
The pipeline expects:

- `X_tr, X_va, X_te` — (N, n_features) `float32`, scaled (StandardScaler-on-train recommended)
- `y_tr, y_va, y_te` — (N,) `int64`, class labels 0..(n_classes-1)

The val split is used for early-stopping inside GP evolution; the test split is
held out for final accuracy / latency reporting.


In [ ]:
import urllib.request
from io import BytesIO
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42

# === Replace this block with your data ===
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
with urllib.request.urlopen(url, timeout=60) as r:
    df = pd.read_csv(BytesIO(r.read()), header=None,
                     names=['preg','glucose','bp','skin','insulin','bmi','pedi','age','class'])
for c in ['glucose','bp','skin','insulin','bmi']:
    df.loc[df[c] == 0, c] = np.nan
for c in df.columns[:-1]: df[c] = df[c].fillna(df[c].median())
X_all = df.drop(columns=['class']).values.astype(np.float32)
y_all = df['class'].values.astype(np.int64)
DATASET_NAME = 'pima'
# =========================================

X_trv, X_te, y_trv, y_te = train_test_split(X_all, y_all, test_size=0.2,
                                              random_state=SEED, stratify=y_all)
X_tr, X_va, y_tr, y_va = train_test_split(X_trv, y_trv, test_size=0.15,
                                            random_state=SEED+1, stratify=y_trv)
sc_X = StandardScaler().fit(X_tr)
X_tr  = sc_X.transform(X_tr).astype(np.float32)
X_va  = sc_X.transform(X_va).astype(np.float32)
X_te  = sc_X.transform(X_te).astype(np.float32)

n_classes = int(y_all.max() + 1)
n_features = X_tr.shape[1]

print(f'Dataset  : {DATASET_NAME}')
print(f'Shape    : train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}')
print(f'Classes  : {n_classes}')
print(f'Features : {n_features}')


## 4. Train the GP ensemble + LR head — ship config

| | |
|---|---|
| `num_members`   | 10 |
| `pop_size`      | 200 |
| `gens`          | 40 |
| `max_depth`     | 6 |
| `class_weight`  | False (accuracy-mode) |
| `boost`         | True |
| `tier`          | 5 |

These are the values that produced the v1 numbers in the stress test:
sub-300 ns single-sample inference, 19 KB artifact, 91% accuracy on Bank, 84%+
on Adult/Telco, win on German.

**Do not change unless you know what you're doing** — lab-tested defaults.


In [ ]:
import time, random
import torch
import gp_ensemble as gpe
from gp_ensemble import evolve_member, majority_vote
from jit_kernel import jit_gpe_score_batch, jit_gpe_lr_predict_batch
from bridge import pack_ensemble_from_gp_ensemble
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
gpe.set_active_ops(5)

cfg = gpe.Config(
    tier=5, num_members=10,
    pop_size=200, gens=40, max_depth=6,
    meta_hidden=64, meta_epochs=300,
    nn_hidden=64, nn_epochs=100,
    boost=True, class_weight=False,    # vanilla v1 ship config
    device='cpu', seed=SEED,
)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)

print(f'Evolving {cfg.num_members} GP members...')
print(f'  config: pop={cfg.pop_size}  gens={cfg.gens}  depth={cfg.max_depth}  '
      f'boost={cfg.boost}  class_weight={cfg.class_weight}')

t0 = time.perf_counter()
members = []
for m in range(1, cfg.num_members + 1):
    weights = None
    if cfg.boost and m > cfg.boost_after and len(members) >= cfg.boost_after:
        with torch.no_grad():
            ep = majority_vote(members, X_tr_t, n_classes)
            wrong = (ep != y_tr_t)
            w = torch.ones(X_tr_t.shape[0], device=cfg.device)
            w[wrong] = cfg.boost_weight
            weights = w / w.mean()
    member = evolve_member(X_tr_t, y_tr_t, cfg, weights=weights, member_num=m)
    members.append(member)
    elapsed = time.perf_counter() - t0
    print(f'  member {m}/{cfg.num_members} done at {elapsed:.1f}s  (size={member.total_size()} nodes)')

evo_seconds = time.perf_counter() - t0
print(f'\nGP evolution: {evo_seconds:.1f}s')


In [ ]:
# Pack to bytecode
pack = pack_ensemble_from_gp_ensemble(members)
op_names = ['FEAT', 'CONST', 'ADD', 'SUB', 'MUL', 'MAX', 'MIN', 'LT', 'CLAMP']
op_counts = np.bincount(pack['instrs'], minlength=9).tolist()
print(f'Packed: m={pack["n_members"]}, c={pack["n_classes"]}, '
      f'total_nodes={pack["total_nodes"]}')
print(f'  op counts: {dict(zip(op_names, op_counts))}')

# Score training data through the GP, train the LR head
stk_tr = jit_gpe_score_batch(X_tr, pack['instrs'], pack['operands'],
                              pack['tree_starts'], pack['n_members'], pack['n_classes'])
stk_te = jit_gpe_score_batch(X_te, pack['instrs'], pack['operands'],
                              pack['tree_starts'], pack['n_members'], pack['n_classes'])

scaler = StandardScaler().fit(stk_tr)
lr = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs').fit(scaler.transform(stk_tr), y_tr)
print(f'\nsklearn-LR test accuracy: {lr.score(scaler.transform(stk_te), y_te):.4f}')


In [ ]:
# Fold scaler weights into LR coefficients (so the .so doesn't have to apply scaler at inference)
def fold_scaler_into_lr(lr, scaler, n_classes):
    W = lr.coef_; b = lr.intercept_
    safe_std = np.where(scaler.scale_ > 1e-8, scaler.scale_, 1.0)
    W_folded = W / safe_std[None, :]
    b_folded = b - (W @ (scaler.mean_ / safe_std))
    if n_classes == 2:
        out_W = np.zeros((2, W_folded.shape[1]), dtype=np.float32)
        out_b = np.zeros(2, dtype=np.float32)
        out_W[1] = W_folded[0]; out_b[1] = b_folded[0]
        return out_W, out_b
    return W_folded.astype(np.float32), b_folded.astype(np.float32)

lr_W, lr_b = fold_scaler_into_lr(lr, scaler, n_classes)
print(f'LR weights shape: {lr_W.shape}, bias shape: {lr_b.shape}')
print(f'  scaler folded into immediates: yes')


## 5. AOT compile to a `.so`

Walks the bytecode symbolically, emits straight-line C with constant-folded LR
weights, compiles via `gcc -O3 -march=native`. Output is a self-contained `.so`
exposing `predict_one`, `predict_batch`, and `predict_time_ns`.


In [ ]:
from lightning_aot import build_aot, emit_c_source

t0 = time.perf_counter()
src_text = emit_c_source(pack, lr_W, lr_b, n_features=n_features)
aot = build_aot(pack, lr_W, lr_b, n_features=n_features)
compile_seconds = time.perf_counter() - t0

so_size = os.path.getsize(aot.so_path)
n_lines = src_text.count(chr(10))
print(f'Generated C   : {len(src_text)} bytes, {n_lines} lines')
print(f'Compile time  : {compile_seconds:.2f}s')
print(f'Compiled .so  : {aot.so_path}')
print(f'Artifact size : {so_size} bytes ({so_size/1024:.1f} KB)')


## 6. Verify — AOT predictions match production JIT bit-for-bit

The AOT path is the deployment artifact, but the Numba JIT path is what we've
validated for accuracy in the lab. Production correctness = the AOT compiles to
predictions identical to the JIT predictions on the held-out test set.

If this cell prints anything other than `100.0000% agreement`, **stop and debug
before deploying** — there's a codegen bug.


In [ ]:
# JIT (production reference) predictions
preds_jit = jit_gpe_lr_predict_batch(
    X_te.astype(np.float32),
    pack['instrs'], pack['operands'], pack['tree_starts'],
    pack['n_members'], pack['n_classes'],
    lr_W, lr_b,
)
# AOT predictions
preds_aot = aot.predict(X_te.astype(np.float32))
# sklearn reference (the original LR before scaler-fold)
preds_sk = lr.predict(scaler.transform(stk_te))

agreement_aot_vs_jit = float((preds_aot == preds_jit).mean())
agreement_aot_vs_sk  = float((preds_aot == preds_sk).mean())
agreement_jit_vs_sk  = float((preds_jit == preds_sk).mean())

acc_aot = float((preds_aot == y_te).mean())

print(f'3-way agreement on {len(y_te)} test samples:')
print(f'  AOT vs JIT       : {agreement_aot_vs_jit*100:.4f}%')
print(f'  AOT vs sklearn-LR: {agreement_aot_vs_sk*100:.4f}%')
print(f'  JIT vs sklearn-LR: {agreement_jit_vs_sk*100:.4f}%')
print(f'\nAOT test accuracy: {acc_aot:.4f}')

assert agreement_aot_vs_jit == 1.0, 'AOT vs JIT mismatch — codegen bug'
print('\n✓ Verification passed.')


## 7. Measure — pure-C single-sample latency

`clock_gettime` from inside the .so over 1 million iterations. Zero Python overhead.
This is the deployment latency — what a customer's API request would see.


In [ ]:
sample = X_te[:1].astype(np.float32)
sample_flat = sample.reshape(-1).astype(np.float32)

ns_per_call = aot.time_ns_per_call(sample_flat, n_warmup=10_000, n_iters=1_000_000)
throughput = 1e9 / ns_per_call

print(f'Pure-C single-sample latency: {ns_per_call:.1f} ns/call ({ns_per_call/1000:.3f} μs)')
print(f'Throughput                  : {throughput:,.0f} predictions/sec')
print(f'Artifact size               : {so_size/1024:.1f} KB')
print(f'GP bytecode nodes           : {pack["total_nodes"]}')


## 8. Bundle for deployment

Pack the `.so` + metadata into a single zip suitable for shipping to a server
or another machine. The metadata captures: feature count, scaler state, accuracy,
training timestamp, source dataset hash. **This is what gets registered in the
inference service's model store.**


In [ ]:
import shutil, zipfile, json, hashlib, datetime

bundle_dir = f'rapid_lightning_v1_{DATASET_NAME}_seed{SEED}'
os.makedirs(bundle_dir, exist_ok=True)

shutil.copy(aot.so_path, f'{bundle_dir}/predict.so')

c_path = aot.so_path.replace('.so', '.c')
if os.path.exists(c_path):
    shutil.copy(c_path, f'{bundle_dir}/predict.c')

# Save the input feature scaler so the customer's runtime knows how to preprocess
np.savez(f'{bundle_dir}/feature_scaler.npz',
          mean=sc_X.mean_, scale=sc_X.scale_)

# Metadata
data_hash = hashlib.sha256(X_all.tobytes()).hexdigest()[:16]
meta = {
    'schema': 'rapid_lightning/v1',
    'dataset': DATASET_NAME,
    'data_sha256_16': data_hash,
    'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'seed': SEED,
    'config': {
        'num_members': cfg.num_members,
        'pop_size':    cfg.pop_size,
        'gens':        cfg.gens,
        'max_depth':   cfg.max_depth,
        'tier':        cfg.tier,
        'class_weight': cfg.class_weight,
        'boost':       cfg.boost,
    },
    'shape': {
        'n_features':   n_features,
        'n_classes':    n_classes,
        'n_train':      int(X_tr.shape[0]),
        'n_test':       int(X_te.shape[0]),
        'gp_total_nodes': int(pack['total_nodes']),
    },
    'performance': {
        'test_accuracy':         float(acc_aot),
        'pure_c_latency_ns':     float(ns_per_call),
        'throughput_pred_per_sec': float(throughput),
        'so_bytes':              int(so_size),
        'evolution_seconds':     float(evo_seconds),
        'compile_seconds':       float(compile_seconds),
    },
    'verification': {
        'aot_vs_jit_agreement_pct': float(agreement_aot_vs_jit * 100),
    },
}
with open(f'{bundle_dir}/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

# README for the recipient
readme = f"""Rapid Lightning v1 — model bundle
==================================

Dataset      : {DATASET_NAME}
Trained at   : {meta['trained_at']}
Test accuracy: {acc_aot:.4f}
Latency      : {ns_per_call:.1f} ns/call (pure C)
Size         : {so_size/1024:.1f} KB

Files
-----
predict.so        - compiled predictor (gcc -O3 -march=native, no Python deps)
predict.c         - generated C source (auditable; do not edit)
feature_scaler.npz- numpy {{mean, scale}} for input preprocessing
metadata.json     - all training + performance metadata

Calling from C
--------------
    int32_t predict_one(const float* X);
    void    predict_batch(const float* X, int32_t N, int32_t n_features, int32_t* out);
    double  predict_time_ns(const float* X, int32_t n_warmup, int32_t n_iters);

Calling from Python
-------------------
    import numpy as np
    from lightning_aot import AOTPredictor   # ship this in your Python wrapper
    scaler = np.load('feature_scaler.npz')
    aot = AOTPredictor('predict.so', n_features={n_features})
    x = np.array([...], dtype=np.float32)
    x = (x - scaler['mean']) / scaler['scale']
    klass = aot.predict_one(x.astype(np.float32))
"""
with open(f'{bundle_dir}/README.txt', 'w') as f:
    f.write(readme)

# Zip
zip_path = f'{bundle_dir}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(bundle_dir):
        for fn in files:
            p = os.path.join(root, fn)
            zf.write(p, os.path.relpath(p, '.'))

print(f'Bundle written: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')
print('Contents:')
for fn in sorted(os.listdir(bundle_dir)):
    sz = os.path.getsize(os.path.join(bundle_dir, fn))
    print(f'  {fn:<22}  {sz:>10,} bytes')


## 9. Done.

You have:

- `predict.so` — the deployable artifact, single file, no deps
- `predict.c` — auditable straight-line C
- `feature_scaler.npz` — input preprocessing state
- `metadata.json` — full provenance

These four files plus the `lightning_aot.py` Python wrapper *(ship this with
your runtime)* are everything an inference server needs to serve predictions.

> Next: stand up the inference service. See `rapid_lightning_v1_design_doc.md` §5 for
> server architecture sketch.
